# Print double tag yields
## Make a nice table of double tag yields

### Load utility functions

In [1]:
gROOT->ProcessLine(".L ../UtilityFunctions.C");

### ```struct``` containing yield with asymmetric uncertainties

In [2]:
struct AsymmYield {
    double Value;
    double PlusError;
    double MinusError;
}

### Function that parses the flavour tag yields

In [3]:
std::map<int, AsymmYield> ParseFlavourTagYield(const std::string &Tag) {
    std::string Filename("/data/bes3/tat/KKpipi_StrongPhase_Analysis_4Bins_20fb/DoubleTagFit/");
    Filename += Tag + "/" + "KKpipi_vs_" + Tag + "_SignalYield.txt";
    const auto FittedYields = ParseParameters(Filename);
    std::map<int, AsymmYield> FitResults;
    for(int Bin = -4; Bin <= 4; Bin++) {
        if(Bin == 0) {
            continue;
        }
        std::string Label("DoubleTag_Flavour_KKpipi_vs_");
        Label += Tag + "_SignalBin" + (Bin > 0 ? "P" : "M");
        Label += std::to_string(TMath::Abs(Bin)) + "_TagBin0_SignalYield";
        double Value = FittedYields.at(Label);
        double PlusError = FittedYields.at(Label + "_high_err");
        double MinusError = -FittedYields.at(Label + "_low_err");
        FitResults.insert({Bin, {Value, PlusError, MinusError}});
    }
    return FitResults;
}

### Function that parses the CP tag yields

In [4]:
std::map<int, AsymmYield> ParseCPTagYield(const std::string &Tag) {
    std::string Filename("/data/bes3/tat/KKpipi_StrongPhase_Analysis_4Bins_20fb/DoubleTagFit/");
    Filename += Tag + "/" + "KKpipi_vs_" + Tag + "_SignalYield.txt";
    const auto FittedYields = ParseParameters(Filename);
    std::map<int, AsymmYield> FitResults;
    for(int Bin = 1; Bin <= 4; Bin++) {
        std::string Label("DoubleTag_CP_KKpipi_vs_");
        Label += Tag + "_SignalBin" + std::to_string(Bin) + "_SignalYield";
        double Value = FittedYields.at(Label);
        double PlusError = FittedYields.at(Label + "_high_err");
        double MinusError = -FittedYields.at(Label + "_low_err");
        FitResults.insert({Bin, {Value, PlusError, MinusError}});
    }
    return FitResults;
}

### Function that parses the SCMB tag yields

In [5]:
std::map<std::pair<int, int>, AsymmYield> ParseSCMBTagYield(const std::string &Tag) {
    std::string Filename("/data/bes3/tat/KKpipi_StrongPhase_Analysis_4Bins_20fb/DoubleTagFit/");
    Filename += Tag + "/" + "KKpipi_vs_" + Tag + "_SignalYield.txt";
    const auto FittedYields = ParseParameters(Filename);
    std::map<std::pair<int, int>, AsymmYield> FitResults;
    for(int TagBin = 1; TagBin <= 8; TagBin++) {
        for(int SignalBin = -4; SignalBin <= 4; SignalBin++) {
            if(SignalBin == 0) {
                continue;
            }
            std::string Label("DoubleTag_SCMB_KKpipi_vs_");
            Label += Tag + "_SignalBin" + (SignalBin > 0 ? "P" : "M");
            Label += std::to_string(TMath::Abs(SignalBin)) + "_TagBin";
            Label += std::to_string(TagBin) + "_SignalYield";
            double Value = FittedYields.at(Label);
            double PlusError = FittedYields.at(Label + "_high_err");
            double MinusError = -FittedYields.at(Label + "_low_err");
            FitResults.insert({{SignalBin, TagBin}, {Value, PlusError, MinusError}});
        }
    }
    return FitResults;
}

### List of flavour tags

In [6]:
std::vector<std::string> FlavourTags{"Kpi", "Kpipi0", "Kpipipi", "KeNu"};

### Load all the yields

In [7]:
std::map<std::string, std::map<int, AsymmYield> > FlavourYields;
for(const auto &Tag : FlavourTags) {
    FlavourYields.insert({Tag, ParseFlavourTagYield(Tag)});
}

### Print table

In [8]:
int BlankSpaces = 8;
std::string Blank = BlankSpaces == 0 ? "" : std::string(BlankSpaces, ' ');
int Width = 25;
for(int Bin = -4; Bin <= 4; Bin++) {
    if(Bin == 0) {
        continue;
    }
    std::cout << Blank;
    std::cout << std::left << std::setw(5);
    std::stringstream ss2;
    ss2 << "$" << Bin << "$";
    std::cout << ss2.str() << " & ";
    for(const auto &Tag : FlavourTags) {
        std::stringstream ss;
        ss << "$";
        ss << std::fixed << std::setprecision(1);
        ss << FlavourYields.at(Tag).at(Bin).Value << "^{+";
        ss << std::fixed << std::setprecision(1);
        ss << FlavourYields.at(Tag).at(Bin).PlusError << "}_{-";
        ss << std::fixed << std::setprecision(1);
        ss << FlavourYields.at(Tag).at(Bin).MinusError << "}$";
        std::cout << std::left << std::setw(Width);
        std::cout << ss.str();
        if(Tag == FlavourTags.back()) {
            std::cout << " \\\\" << "\n";
        } else {
            std::cout << " & ";
        }
    }
}

        $-4$  & $61.2^{+8.5}_{-7.8}$      & $117.9^{+11.7}_{-11.1}$   & $84.1^{+10.1}_{-9.5}$     & $60.4^{+7.8}_{-7.2}$      \\
        $-3$  & $279.0^{+17.5}_{-16.8}$   & $442.9^{+22.6}_{-21.9}$   & $268.3^{+17.7}_{-17.0}$   & $175.9^{+13.9}_{-13.2}$   \\
        $-2$  & $209.5^{+15.1}_{-14.5}$   & $365.4^{+20.2}_{-19.6}$   & $218.5^{+15.8}_{-15.2}$   & $126.4^{+11.6}_{-11.0}$   \\
        $-1$  & $79.7^{+9.8}_{-9.1}$      & $165.2^{+14.0}_{-13.3}$   & $101.0^{+11.2}_{-10.5}$   & $67.8^{+8.5}_{-8.0}$      \\
        $1$   & $231.2^{+16.0}_{-15.3}$   & $436.5^{+21.9}_{-21.2}$   & $288.1^{+17.9}_{-17.2}$   & $164.8^{+13.4}_{-12.8}$   \\
        $2$   & $496.9^{+23.1}_{-22.4}$   & $931.7^{+31.8}_{-31.1}$   & $566.5^{+24.8}_{-24.1}$   & $318.1^{+18.8}_{-18.2}$   \\
        $3$   & $524.3^{+23.6}_{-22.9}$   & $999.4^{+33.1}_{-32.5}$   & $576.3^{+25.2}_{-24.6}$   & $378.4^{+20.2}_{-19.5}$   \\
        $4$   & $217.6^{+15.1}_{-14.4}$   & $348.9^{+19.8}_{-19.1}$   & $235.6^{+16.4}_{-15.7}$  

### List of CP tags

In [9]:
std::vector<std::string> CPTags{
    "KK",
    "KKPartReco",
    "pipi",
    "pipipi0",
    "KSpi0pi0",
    "KLpi0",
    "KSpi0",
    "KSpi0PartReco",
    "KSeta",
    "KSetaPrimepipieta",
    "KSetaPrimerhogamma",
    "KSpipipi0"
};

### Load all the yields

In [10]:
std::map<std::string, std::map<int, AsymmYield> > CPYields;
for(const auto &Tag : CPTags) {
    CPYields.insert({Tag, ParseCPTagYield(Tag)});
}

### Print table

In [11]:
int BlankSpaces = 8;
std::string Blank = BlankSpaces == 0 ? "" : std::string(BlankSpaces, ' ');
int Width = 25;
for(const auto &Tag : CPTags) {
    std::cout << Blank;
    std::cout << std::left << std::setw(35);
    std::cout << GetTagName(Tag) << " & ";
    for(int Bin = 1; Bin <= 4; Bin++) {
        std::stringstream ss;
        ss << "$";
        ss << std::fixed << std::setprecision(1);
        ss << CPYields.at(Tag).at(Bin).Value << "^{+";
        ss << std::fixed << std::setprecision(1);
        ss << CPYields.at(Tag).at(Bin).PlusError << "}_{-";
        ss << std::fixed << std::setprecision(1);
        ss << CPYields.at(Tag).at(Bin).MinusError << "}$";
        std::cout << std::left << std::setw(Width);
        std::cout << ss.str();
        if(Bin == 4) {
            std::cout << " \\\\" << "\n";
        } else {
            std::cout << " & ";
        }
    }
}

        $\kaonp\kaonm$                      & $46.6^{+8.1}_{-7.4}$      & $20.6^{+6.0}_{-5.3}$      & $22.5^{+6.1}_{-5.4}$      & $46.7^{+8.0}_{-7.3}$      \\
        $\kaonp\kaonm$ part. reco.          & $42.6^{+9.2}_{-8.4}$      & $34.6^{+8.9}_{-8.2}$      & $22.1^{+10.0}_{-9.0}$     & $35.1^{+7.6}_{-6.9}$      \\
        $\pip\pim$                          & $11.3^{+5.2}_{-4.5}$      & $3.0^{+4.5}_{-3.0}$       & $5.9^{+5.2}_{-4.4}$       & $11.6^{+4.9}_{-4.2}$      \\
        $\pip\pim\piz$                      & $74.1^{+12.7}_{-12.3}$    & $68.9^{+11.6}_{-11.1}$    & $70.6^{+11.6}_{-11.4}$    & $80.4^{+12.8}_{-12.4}$    \\
        $\kshort\piz\piz$                   & $15.1^{+4.9}_{-4.2}$      & $6.4^{+3.2}_{-2.6}$       & $14.6^{+4.8}_{-4.2}$      & $14.0^{+4.4}_{-3.8}$      \\
        $\klong\piz$                        & $31.3^{+8.4}_{-7.8}$      & $21.9^{+4.9}_{-4.9}$      & $13.6^{+3.7}_{-3.8}$      & $19.8^{+7.2}_{-6.5}$      \\
        $\kshort\piz$                       & 

### List SCMB tags

In [12]:
std::vector<std::string> SCMBTags{
    "KSpipi",
    "KSpipiPartReco",
    "KLpipi"
};

### Load all the yields

In [13]:
std::map<std::string, std::map<std::pair<int, int>, AsymmYield> > SCMBYields;
for(const auto &Tag : SCMBTags) {
    SCMBYields.insert({Tag, ParseSCMBTagYield(Tag)});
}

### Print table

In [14]:
int BlankSpaces = 8;
std::string Blank = BlankSpaces == 0 ? "" : std::string(BlankSpaces, ' ');
int Width = 25;
for(const auto &Tag : SCMBTags) {
    std::cout << Tag << "\n";
    for(int SignalBin = -4; SignalBin <= 4; SignalBin++) {
        if(SignalBin == 0) {
            continue;
        }
        std::cout << Blank;
        std::cout << std::left << std::setw(4);
        std::stringstream ss;
        ss << "$" << SignalBin << "$";
        std::cout << ss.str() << " & ";
        for(int TagBin = 1; TagBin <= 8; TagBin++) {
            std::stringstream ss2;
            ss2 << "$";
            ss2 << std::fixed << std::setprecision(1);
            ss2 << SCMBYields.at(Tag).at({SignalBin, TagBin}).Value << "^{+";
            ss2 << std::fixed << std::setprecision(1);
            ss2 << SCMBYields.at(Tag).at({SignalBin, TagBin}).PlusError << "}_{-";
            ss2 << std::fixed << std::setprecision(1);
            ss2 << SCMBYields.at(Tag).at({SignalBin, TagBin}).MinusError << "}$";
            std::cout << std::left << std::setw(Width);
            std::cout << ss2.str();
            if(TagBin == 8) {
                std::cout << " \\\\" << "\n";
            } else {
                std::cout << " & ";
            }
        }
    }
}

KSpipi
        $-4$ & $9.6^{+3.7}_{-3.1}$       & $9.2^{+3.6}_{-2.9}$       & $5.8^{+2.7}_{-2.0}$       & $1.6^{+1.8}_{-1.1}$       & $3.7^{+2.4}_{-1.7}$       & $1.0^{+1.3}_{-0.7}$       & $0.5^{+1.4}_{-0.5}$       & $7.2^{+3.2}_{-2.5}$       \\
        $-3$ & $7.6^{+3.5}_{-2.9}$       & $8.1^{+3.3}_{-2.7}$       & $6.1^{+3.3}_{-2.6}$       & $8.6^{+3.4}_{-2.7}$       & $21.0^{+5.2}_{-4.6}$      & $9.0^{+3.4}_{-2.7}$       & $4.9^{+2.6}_{-1.9}$       & $6.7^{+3.1}_{-2.4}$       \\
        $-2$ & $9.6^{+3.8}_{-3.1}$       & $2.0^{+2.1}_{-1.4}$       & $6.4^{+3.4}_{-2.7}$       & $11.9^{+3.8}_{-3.1}$      & $19.0^{+5.1}_{-4.4}$      & $4.8^{+2.9}_{-2.2}$       & $2.4^{+2.2}_{-1.5}$       & $7.1^{+3.2}_{-2.5}$       \\
        $-1$ & $8.0^{+3.3}_{-2.6}$       & $3.9^{+2.3}_{-1.7}$       & $2.0^{+1.9}_{-1.2}$       & $1.0^{+1.3}_{-0.7}$       & $3.9^{+2.3}_{-1.7}$       & $4.3^{+2.6}_{-1.9}$       & $4.8^{+2.9}_{-2.2}$       & $4.3^{+2.8}_{-2.1}$       \\
        $1$  & $17.1^{+4.7}_{-4.1